# Rekomendasi Materi

In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 7.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sentence_transformers import SentenceTransformer
import re

df_res = pd.read_csv('mdl_resource.csv')
df_hvp = pd.read_csv('mdl_hvp.csv')
df_url = pd.read_csv('mdl_url.csv')
df_assign = pd.read_csv('mdl_assign.csv')
df_forum = pd.read_csv('mdl_forum.csv')

res_clean = df_res[['id', 'name', 'course']].copy()
res_clean['material_type'] = 'Materi/Dokumen'

hvp_clean = df_hvp[['id', 'name', 'course']].copy()
hvp_clean['material_type'] = 'Materi Interaktif H5P'

url_clean = df_url[['id', 'name', 'course']].copy()
url_clean['material_type'] = 'Link Web/Video'

assign_clean = df_assign[['id', 'name', 'course']].copy()
assign_clean['material_type'] = 'Penugasan'

forum_clean = df_forum[['id', 'name', 'course']].copy()
forum_clean['material_type'] = 'Forum Diskusi'

katalog_final = pd.concat([res_clean, hvp_clean, url_clean, assign_clean, forum_clean], ignore_index=True)
def preprocess_text(text):
    text = str(text).lower()
    kamus_slang = {
        "pake": "pakai",
        "yg": "yang",
        "ga": "tidak",
        "bisa": "bisa",
        "gmn": "bagaimana",
        "knp": "kenapa",
        "actuator": "aktuator"
    }

    words = text.split()
    normalized_words = [kamus_slang.get(word, word) for word in words]
    text = ' '.join(normalized_words)
    return text

katalog_final = katalog_final.dropna(subset=['name', 'course']).reset_index(drop=True)
katalog_final['clean_name'] = katalog_final['name'].apply(preprocess_text)
katalog_final['text_for_embedding'] = katalog_final['clean_name'] + " [Tipe: " + katalog_final['material_type'] + "]"


model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)
embeddings = model.encode(katalog_final['text_for_embedding'].tolist(), show_progress_bar=True)

recommendation_data = {
    'catalog': katalog_final.drop(columns=['text_for_embedding'], errors='ignore'),
    'embeddings': embeddings,
    'model_config': {'model_name': model_name}
}

joblib.dump(recommendation_data, 'smart_lmsx_recom_engine.pkl')
print("Katalog yang bersih & vektor SBERT telah tersimpan.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Katalog yang bersih & vektor SBERT telah tersimpan.


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import pandas as pd

recom_data = joblib.load('smart_lmsx_recom_engine.pkl')
katalog_df = recom_data['catalog']
katalog_embeddings = recom_data['embeddings']
model = SentenceTransformer(recom_data['model_config']['model_name'])


course_mapping = {
    5322: 5321,
}

target_course = 5322
keluhan_mahasiswa = "Bagaimana cara kerja sensor suhu?"

print(f"Akses Awal dari Course ID : {target_course}")

search_courses = [target_course]
if target_course in course_mapping:
    teori_course = course_mapping[target_course]
    search_courses.append(teori_course)
    print(f"-> Terdeteksi Kelas Praktikum. Ruang pencarian digabungkan dengan Kelas Teori ID: {teori_course}")
else:
    print(f"-> Menggunakan Kelas Reguler ID: {target_course}")

print(f"Input Keluhan             : '{keluhan_mahasiswa}'\n")

katalog_df['course'] = pd.to_numeric(katalog_df['course'], errors='coerce').fillna(0).astype(int)

indeks_valid = katalog_df.index[katalog_df['course'].isin(search_courses)].tolist()

if len(indeks_valid) == 0:
    print("Peringatan: Tidak ada materi yang ditemukan untuk ID kursus ini di dalam katalog.")
else:
    katalog_lokal_df = katalog_df.iloc[indeks_valid].copy()
    katalog_lokal_embeddings = katalog_embeddings[indeks_valid]

    keluhan_mahasiswa_bersih = preprocess_text(keluhan_mahasiswa)
    keluhan_vektor = model.encode([keluhan_mahasiswa_bersih])

    skor_kemiripan = cosine_similarity(keluhan_vektor, katalog_lokal_embeddings)[0]
    katalog_lokal_df['cosine_score'] = skor_kemiripan

    top_rekomendasi = katalog_lokal_df.sort_values(by='cosine_score', ascending=False).head(3)

    for index, row in top_rekomendasi.iterrows():
        print(f"Rekomendasi Materi : {row['name']} ({row['material_type']}) - [Asal Course: {row['course']}]")
        print(f"Skor Cosine        : {row['cosine_score']:.4f}\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Akses Awal dari Course ID : 5322
-> Terdeteksi Kelas Praktikum. Ruang pencarian digabungkan dengan Kelas Teori ID: 5321
Input Keluhan             : 'Bagaimana cara kerja sensor suhu?'

Rekomendasi Materi : Studi Kasus: Runing LED dengan sensor suhu (Forum Diskusi) - [Asal Course: 5322]
Skor Cosine        : 0.8300

Rekomendasi Materi : Studi Kasus: Implementasi sensor suhu dan asap (Forum Diskusi) - [Asal Course: 5322]
Skor Cosine        : 0.7256

Rekomendasi Materi : Library Dallas Temperature (Materi/Dokumen) - [Asal Course: 5321]
Skor Cosine        : 0.6803

